In [ ]:
import os
import json
import glob
from tqdm import tqdm

import numpy as np
from skimage import io, img_as_ubyte
from skimage.color import gray2rgb

from google.colab import drive
drive.mount('/content/drive')

# --- adjust these as needed ---
BASE_DIR          = '/content/drive/MyDrive/biotech/Retina_Lab/maskrcnn_approach'
OUTPUT_PATCH_ROOT = '/content/drive/MyDrive/biotech/Retina_Lab/patches'
FINAL_JSON_PATH   = '/content/drive/MyDrive/biotech/Retina_Lab/all_patches_annotations.json'

PATCH_SIZE     = (256, 256)   # height, width
STRIDE         = 128          # patch step (for overlap)
IOU_THRESHOLD  = 0.5          # min IOU vs original mask to keep a patch
MIN_MASK_AREA  = 50           # min pixels in-crop for a mask
OVERLAY_ALPHA  = 0.5          # transparency for overlay
# --------------------------------

class NumpyEncoder(json.JSONEncoder):
    """JSON encoder for NumPy types."""
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

def iou(mask1, mask2):
    inter = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return inter / union if union > 0 else 0.0

def extract_masks(seg_array):
    """
    Turn a segmentation array into a list of 2D boolean masks.
    Supports either (H,W,instances) stacks or (H,W) label maps.
    """
    masks = []
    if isinstance(seg_array, np.ndarray) and seg_array.ndim == 3:
        # a stack of one-hot masks
        for i in range(seg_array.shape[2]):
            m = seg_array[..., i].astype(bool)
            if m.sum() > 0:
                masks.append(m)
    elif isinstance(seg_array, np.ndarray) and seg_array.ndim == 2:
        # discrete labels
        for label in np.unique(seg_array):
            if label == 0:
                continue
            m = (seg_array == label)
            if m.sum() > 0:
                masks.append(m)
    else:
        raise ValueError(f"Unexpected segmentation array shape {getattr(seg_array,'shape',None)}")
    return masks

def create_patches(img, masks, patch_size, stride, min_mask_area=0, iou_thresh=0):
    H, W = img.shape[:2]
    ph, pw = patch_size
    annos = []

    for y0 in range(0, H, stride):
        for x0 in range(0, W, stride):
            y1, x1 = min(y0+ph, H), min(x0+pw, W)
            patch_img = img[y0:y1, x0:x1]
            kept_masks, bboxes = [], []

            for m in masks:
                # crop mask
                m_patch = m[y0:y1, x0:x1]
                if m_patch.sum() < min_mask_area:
                    continue
                # pad back for IOU
                padded = np.zeros_like(m)
                padded[y0:y1, x0:x1] = m_patch
                if iou(m, padded) < iou_thresh:
                    continue
                ys, xs = np.where(m_patch)
                kept_masks.append(m_patch)
                bboxes.append([int(ys.min()), int(xs.min()), int(ys.max()), int(xs.max())])

            if kept_masks:
                annos.append({
                    'patch_img': patch_img,
                    'masks': kept_masks,
                    'bboxes': bboxes,
                    'source_bbox': [y0, x0, y1, x1],
                })

    return annos

def save_overlay(patch_img, masks, out_path, alpha=0.5):
    """
    Save an RGB overlay (masks in red) on the patch.
    """
    # ensure RGB base
    base = gray2rgb(patch_img) if patch_img.ndim == 2 else patch_img.copy()
    overlay = base.astype(np.float32)

    combined = np.zeros(base.shape[:2], dtype=bool)
    for m in masks:
        combined |= m

    # apply red overlay in 0–255 space
    overlay[..., 0][combined] = overlay[..., 0][combined] * (1 - alpha) + 255 * alpha
    overlay[..., 1][combined] *= (1 - alpha)
    overlay[..., 2][combined] *= (1 - alpha)

    # clamp and convert to uint8
    overlay_uint8 = np.clip(overlay, 0, 255).astype(np.uint8)
    io.imsave(out_path, overlay_uint8)

# ——— Main loop ———
all_images = sorted(glob.glob(os.path.join(BASE_DIR, '*.png')))
annotations = []

for img_path in tqdm(all_images, desc='Processing images'):
    fname = os.path.basename(img_path)
    stem, _ = os.path.splitext(fname)
    seg_path = os.path.join(BASE_DIR, f"{stem}_seg.npy")
    if not os.path.exists(seg_path):
        print(f"⚠️ Missing {stem}_seg.npy, skipping.")
        continue

    img = io.imread(img_path)
    seg_raw = np.load(seg_path, allow_pickle=True)
    # if array is 0-d object, unpack
    if isinstance(seg_raw, np.ndarray) and seg_raw.shape == ():
        seg_raw = seg_raw.item()
    # if Mask R‑CNN dict format
    if isinstance(seg_raw, dict) and 'masks' in seg_raw:
        seg_array = seg_raw['masks']
    else:
        seg_array = seg_raw

    masks = extract_masks(seg_array)

    patches = create_patches(
        img, masks,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
        min_mask_area=MIN_MASK_AREA,
        iou_thresh=IOU_THRESHOLD
    )

    out_dir = os.path.join(OUTPUT_PATCH_ROOT, stem)
    os.makedirs(out_dir, exist_ok=True)

    for idx, p in enumerate(patches):
        patch_name   = f"{stem}_patch{idx:03d}.png"
        overlay_name = f"{stem}_patch{idx:03d}_overlay.png"
        patch_path   = os.path.join(out_dir, patch_name)
        overlay_path = os.path.join(out_dir, overlay_name)

        io.imsave(patch_path, img_as_ubyte(p['patch_img']))
        save_overlay(p['patch_img'], p['masks'], overlay_path, alpha=OVERLAY_ALPHA)

        annotations.append({
            'source_image': fname,
            'patch_image': patch_path,
            'overlay_image': overlay_path,
            'source_bbox': p['source_bbox'],
            'bboxes': p['bboxes'],
            'mask_counts': [int(m.sum()) for m in p['masks']],
        })

# dump JSON
with open(FINAL_JSON_PATH, 'w') as f:
    json.dump(annotations, f, cls=NumpyEncoder, indent=2)

print(f"✅ Finished {len(annotations)} patches. Annotations at:\n  {FINAL_JSON_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Processing images:   0%|          | 0/6 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/Copy of C1-C30000/Copy of C1-C30000_patch025.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/Copy of C1-C30000/Copy of C1-C30000_patch030.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/Copy of C1-C30000/Copy of C1-C30000_patch031.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/Copy of C1-C30000/Copy of C1-C30000_patch032.png is a low contrast image
  return func(*args, **kwar

✅ Finished 176 patches. Annotations at:
  /content/drive/MyDrive/biotech/Retina_Lab/all_patches_annotations.json


## Write the COCO Json file

In [ ]:
# ─── 0) Install dependencies ───────────────────────────────────────────
!pip install pycocotools

# ─── 1) Imports & Drive mount ──────────────────────────────────────────
import os, json, glob
from tqdm import tqdm

import numpy as np
from skimage import io, img_as_ubyte
from skimage.color import gray2rgb
from pycocotools import mask as mask_utils

from google.colab import drive
drive.mount('/content/drive')

# ─── 2) Paths & parameters ─────────────────────────────────────────────
BASE_DIR          = '/content/drive/MyDrive/biotech/Retina_Lab/maskrcnn_approach'
OUTPUT_PATCH_ROOT = '/content/drive/MyDrive/biotech/Retina_Lab/patches'
COCO_JSON_PATH    = '/content/drive/MyDrive/biotech/Retina_Lab/coco_annotations.json'

PATCH_SIZE    = (256, 256)  # (h, w)
STRIDE        = 128         # for overlapping patches
MIN_MASK_AREA = 50          # minimum pixels in a patch to keep mask

# Channel-to-category mapping
CHANNEL2CAT = {'C1': 0, 'C2': 2, 'C3': 1}
CATEGORIES = [
    {"id": 0, "name": "other"},
    {"id": 1, "name": "rbc"},
    {"id": 2, "name": "cbc"},
]

# ─── 3) Helpers ────────────────────────────────────────────────────────
def extract_masks(seg_array):
    """Split a (H,W,instances) or (H,W) array into a list of boolean masks."""
    masks = []
    if isinstance(seg_array, np.ndarray) and seg_array.ndim == 3:
        for i in range(seg_array.shape[2]):
            m = seg_array[..., i].astype(bool)
            if m.sum() > 0:
                masks.append(m)
    elif isinstance(seg_array, np.ndarray) and seg_array.ndim == 2:
        for lbl in np.unique(seg_array):
            if lbl == 0:
                continue
            m = (seg_array == lbl)
            if m.sum() > 0:
                masks.append(m)
    else:
        raise ValueError(f"Unexpected seg array shape: {getattr(seg_array, 'shape', None)}")
    return masks

def make_patches(img, masks, patch_size, stride):
    """Yield dicts {patch_img, masks, bboxes} for every overlapping patch."""
    H, W = img.shape[:2]
    ph, pw = patch_size
    for y0 in range(0, H, stride):
        for x0 in range(0, W, stride):
            y1, x1 = min(y0+ph, H), min(x0+pw, W)
            patch = img[y0:y1, x0:x1]
            keep_m, bboxes = [], []
            for m in masks:
                m_patch = m[y0:y1, x0:x1]
                if m_patch.sum() < MIN_MASK_AREA:
                    continue
                ys, xs = np.where(m_patch)
                ymin, xmin, ymax, xmax = ys.min(), xs.min(), ys.max(), xs.max()
                keep_m.append(m_patch)
                # COCO bbox format: [xmin, ymin, width, height]
                bboxes.append([int(xmin), int(ymin), int(xmax-xmin+1), int(ymax-ymin+1)])
            if keep_m:
                yield {
                    "patch_img": patch,
                    "masks": keep_m,
                    "bboxes": bboxes,
                }

def save_patch(patch_img, out_dir, img_id):
    """Save patch PNG and return its relative path for JSON."""
    filename = f"patch_{img_id:06d}.png"
    full = os.path.join(out_dir, filename)
    io.imsave(full, img_as_ubyte(patch_img))
    # return path relative to the JSON file location
    return os.path.relpath(full, start=os.path.dirname(COCO_JSON_PATH))

# ─── 4) Build COCO JSON ─────────────────────────────────────────────────
coco = {"images": [], "annotations": [], "categories": CATEGORIES}
img_id = 0
ann_id = 0

for img_path in tqdm(sorted(glob.glob(os.path.join(BASE_DIR, "*.png")))):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    channel = stem.split("_")[0]     # e.g. "C1"
    category_id = CHANNEL2CAT.get(channel, 0)

    seg_path = os.path.join(BASE_DIR, f"{stem}_seg.npy")
    if not os.path.exists(seg_path):
        print(f"⚠️ Missing {stem}_seg.npy; skipping.")
        continue

    # load image + segmentation
    img = io.imread(img_path)
    seg_raw = np.load(seg_path, allow_pickle=True)
    # unwrap 0-d object arrays
    if isinstance(seg_raw, np.ndarray) and seg_raw.shape == ():
        seg_raw = seg_raw.item()
    # if stored as dict with 'masks' key
    if isinstance(seg_raw, dict) and 'masks' in seg_raw:
        seg_array = seg_raw['masks']
    else:
        seg_array = seg_raw

    # extract instance masks
    masks = extract_masks(seg_array)

    # prepare output subfolder
    out_dir = os.path.join(OUTPUT_PATCH_ROOT, stem)
    os.makedirs(out_dir, exist_ok=True)

    # generate patches and their entries
    for patch in make_patches(img, masks, PATCH_SIZE, STRIDE):
        img_id += 1
        rel_path = save_patch(patch["patch_img"], out_dir, img_id)

        # image entry
        h, w = patch["patch_img"].shape[:2]
        coco["images"].append({
            "id": img_id,
            "file_name": rel_path,
            "height": h,
            "width": w
        })

        # annotations for each instance in this patch
        for m, bbox in zip(patch["masks"], patch["bboxes"]):
            rle = mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))
            rle["counts"] = rle["counts"].decode("ascii")
            ann_id += 1
            coco["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": category_id,
                "bbox": bbox,
                "area": int(mask_utils.area(rle)),
                "segmentation": rle,
                "iscrowd": 0
            })

# write final COCO JSON
with open(COCO_JSON_PATH, "w") as f:
    json.dump(coco, f, indent=2)

print(f"✅ Wrote COCO JSON with {len(coco['images'])} images and {len(coco['annotations'])} annotations:")
print("   ", COCO_JSON_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


  0%|          | 0/6 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/C1-C30000/patch_000026.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/C1-C30000/patch_000027.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/C1-C30000/patch_000028.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/C1-C30000/patch_000033.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDri

✅ Wrote COCO JSON with 197 images and 2264 annotations:
    /content/drive/MyDrive/biotech/Retina_Lab/coco_annotations.json


In [ ]:
# ─── 0) Install dependencies ───────────────────────────────────────────
!pip install pycocotools

# ─── 1) Imports & Drive mount ──────────────────────────────────────────
import os, json, glob
from tqdm import tqdm

import numpy as np
from skimage import io, img_as_ubyte
from pycocotools import mask as mask_utils

from google.colab import drive
drive.mount('/content/drive')

# ─── 2) Paths & parameters ─────────────────────────────────────────────
BASE_DIR          = '/content/drive/MyDrive/biotech/Retina_Lab/maskrcnn_approach'
OUTPUT_PATCH_ROOT = '/content/drive/MyDrive/biotech/Retina_Lab/patches'
COCO_JSON_PATH    = '/content/drive/MyDrive/biotech/Retina_Lab/coco_annotations.json'

PATCH_SIZE    = (256, 256)  # (height, width)
STRIDE        = 128         # overlap stride
MIN_MASK_AREA = 50          # drop tiny mask fragments

# Channel‐to‐category mapping: C1→0(other), C2→1(rbc), C3→2(cbc)
CHANNEL2CAT = {'C1': 0, 'C2': 1, 'C3': 2}
CATEGORIES = [
    {"id": 0, "name": "other"},
    {"id": 1, "name": "rbc"},
    {"id": 2, "name": "cbc"},
]

# ─── 3) Helper functions ────────────────────────────────────────────────
def extract_masks(seg_array):
    """
    Turn a (H,W,instances) array or (H,W) label map into a list of boolean masks.
    """
    masks = []
    if isinstance(seg_array, np.ndarray) and seg_array.ndim == 3:
        for i in range(seg_array.shape[2]):
            m = seg_array[..., i].astype(bool)
            if m.sum() > 0:
                masks.append(m)
    elif isinstance(seg_array, np.ndarray) and seg_array.ndim == 2:
        for lbl in np.unique(seg_array):
            if lbl == 0:
                continue
            m = (seg_array == lbl)
            if m.sum() > 0:
                masks.append(m)
    else:
        raise ValueError(f"Unexpected seg array shape: {getattr(seg_array,'shape',None)}")
    return masks

def make_patches(img, masks, patch_size, stride):
    """
    Yield dicts for each overlapping patch:
      { "patch_img", "masks", "bboxes" }
    """
    H, W = img.shape[:2]
    ph, pw = patch_size
    for y0 in range(0, H, stride):
        for x0 in range(0, W, stride):
            y1, x1 = min(y0+ph, H), min(x0+pw, W)
            patch_img = img[y0:y1, x0:x1]
            kept_masks, bboxes = [], []
            for m in masks:
                m_patch = m[y0:y1, x0:x1]
                if m_patch.sum() < MIN_MASK_AREA:
                    continue
                ys, xs = np.where(m_patch)
                ymin, xmin, ymax, xmax = ys.min(), xs.min(), ys.max(), xs.max()
                kept_masks.append(m_patch)
                # COCO bbox: [xmin, ymin, width, height]
                bboxes.append([
                    int(xmin),
                    int(ymin),
                    int(xmax - xmin + 1),
                    int(ymax - ymin + 1)
                ])
            if kept_masks:
                yield {
                    "patch_img": patch_img,
                    "masks": kept_masks,
                    "bboxes": bboxes
                }

def save_patch(patch_img, out_dir, img_id):
    """
    Save the patch to disk and return its path relative to the JSON file.
    """
    filename = f"patch_{img_id:06d}.png"
    full_path = os.path.join(out_dir, filename)
    io.imsave(full_path, img_as_ubyte(patch_img))
    return os.path.relpath(full_path, start=os.path.dirname(COCO_JSON_PATH))

# ─── 4) Build COCO JSON ─────────────────────────────────────────────────
coco = {
    "images": [],
    "annotations": [],
    "categories": CATEGORIES
}
img_id = 0
ann_id = 0

for img_path in tqdm(sorted(glob.glob(os.path.join(BASE_DIR, "*.png")))):
    # derive stem and channel
    stem = os.path.splitext(os.path.basename(img_path))[0]
    channel = stem.split("_")[0]        # e.g. "C1"
    category_for_patch = CHANNEL2CAT.get(channel, 0)

    seg_path = os.path.join(BASE_DIR, f"{stem}_seg.npy")
    if not os.path.exists(seg_path):
        print(f"⚠️ Missing {stem}_seg.npy; skipping.")
        continue

    # load image + segmentation
    img = io.imread(img_path)
    seg_raw = np.load(seg_path, allow_pickle=True)
    # unwrap zero‐dim object arrays
    if isinstance(seg_raw, np.ndarray) and seg_raw.shape == ():
        seg_raw = seg_raw.item()
    # if dict with 'masks' key
    if isinstance(seg_raw, dict) and "masks" in seg_raw:
        seg_array = seg_raw["masks"]
    else:
        seg_array = seg_raw

    masks = extract_masks(seg_array)

    # prepare output folder
    out_dir = os.path.join(OUTPUT_PATCH_ROOT, stem)
    os.makedirs(out_dir, exist_ok=True)

    # generate & save patches + annotations
    for patch in make_patches(img, masks, PATCH_SIZE, STRIDE):
        img_id += 1
        rel_path = save_patch(patch["patch_img"], out_dir, img_id)

        # add image entry
        h, w = patch["patch_img"].shape[:2]
        coco["images"].append({
            "id": img_id,
            "file_name": rel_path,
            "height": h,
            "width": w
        })

        # add one annotation per instance mask
        for m, bbox in zip(patch["masks"], patch["bboxes"]):
            # encode mask as COCO RLE
            rle = mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))
            rle["counts"] = rle["counts"].decode("ascii")

            ann_id += 1
            coco["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": category_for_patch,
                "bbox": bbox,
                "area": int(mask_utils.area(rle)),
                "segmentation": rle,
                "iscrowd": 0
            })

# ─── 5) Write out JSON ───────────────────────────────────────────────────
with open(COCO_JSON_PATH, "w") as f:
    json.dump(coco, f, indent=2)

print(f"✅ COCO JSON saved to {COCO_JSON_PATH}")
print(f"   {len(coco['images'])} images, {len(coco['annotations'])} annotations")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


  0%|          | 0/6 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/C1-C30000/patch_000026.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/C1-C30000/patch_000027.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/C1-C30000/patch_000028.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDrive/biotech/Retina_Lab/patches/C1-C30000/patch_000033.png is a low contrast image
  return func(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/skimage/_shared/utils.py:328: UserWarning: /content/drive/MyDri

✅ COCO JSON saved to /content/drive/MyDrive/biotech/Retina_Lab/coco_annotations.json
   197 images, 2264 annotations
